# **Vietnamese Text Summarization - Progressive Training (Kaggle)**

Thí nghiệm gradual unfreezing:
- **Phase 1**: Full fine-tune trên dữ liệu tổng quát
- **Phase 2a**: Freeze encoder, chỉ train decoder trên dữ liệu chuyên ngành (Medical)
- **Phase 2b**: Unfreeze toàn bộ, train với LR rất thấp trên dữ liệu chuyên ngành

Dữ liệu không nằm trong Git. Notebook tự mount và kiểm tra hai Kaggle Dataset:
- `sumarization_phase_1` (Kaggle có thể đổi slug thành `sumarization-phase-1`)
- `sumarization_phase_2`; tên rút gọn `phase_2` cũng được hỗ trợ

> **Lưu ý:** Đây là một hướng thí nghiệm, không mặc định là phương án giữ kiến thức Phase 1 tốt nhất. Luôn xem retention check ở cuối notebook; với deployment tách domain, `Phase 1 + LoRA Phase 2` an toàn hơn.

### Hướng dẫn sử dụng
1. Add cả hai Dataset ở trên vào Kaggle Notebook.
2. Bật GPU (T4 x2 hoặc P100) và Internet cho cell cài đặt/model lần đầu.
3. Mặc định `RUN_MODE="smoke"` để kiểm tra luồng nhanh. Khi thành công, đổi thành `"full"`, Restart Session rồi Run All.
4. Không dùng test split để chọn checkpoint.

In [ ]:
from __future__ import annotations
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"
os.environ["WANDB_CONSOLE"] = "off"
os.environ["GIT_TERMINAL_PROMPT"] = "0"
os.environ["PIP_NO_INPUT"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
print("🔕 Non-interactive mode: W&B disabled; Git/Pip prompts disabled.")

In [ ]:
REPO_URL = "https://github.com/dungcony/sumarization.git"

if os.path.exists(".git") and "sumarization" in os.getcwd():
    print("Đang cập nhật code...")
    !git pull
else:
    print("Đang tải mã nguồn...")
    !git clone {REPO_URL} sumarization
    %cd sumarization

In [ ]:
%pip install -e .

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import gc
import time
import threading
import csv
import re
from dataclasses import replace
from pathlib import Path
import torch

from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    TrainerCallback,
)

from src.config import load_config, apply_overrides, config_to_dict
from src.data import load_and_preprocess, load_dataset_from_files
from src.evaluator import build_compute_metrics
from src.model import (
    enable_gradient_checkpointing,
    freeze_encoder,
    load_model,
    load_tokenizer,
)
from src.trainer import build_training_args
from src.utils import (
    save_json,
    set_seed,
    setup_logger,
)

logger = setup_logger("notebook")
print("✅ Import thành công!")

## Cấu hình và kiểm tra Kaggle Dataset

In [ ]:
KAGGLE_DATASET_ALIASES = {
    "phase_1": ("sumarization_phase_1", "phase_1"),
    "phase_2": ("sumarization_phase_2", "phase_2"),
}

# Chỉ điền khi cấu trúc mount của bạn khác chuẩn. Root phải chứa trực tiếp
# train_*, validation_* và test_* (CSV/Parquet).
MANUAL_DATA_ROOTS = {"phase_1": "", "phase_2": ""}
RUN_MODE = "smoke"  # "smoke" để test luồng; đổi thành "full" để train thật.
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE phải là 'smoke' hoặc 'full'")
SMOKE_TEST = RUN_MODE == "smoke"
SMOKE_RUN_TAG = "smoke_01"  # Đổi tag khi muốn chạy lại một smoke run sạch.
SMOKE_MAX_STEPS = 2
SMOKE_TRAIN_SAMPLES = 16
SMOKE_EVAL_SAMPLES = 8
SMOKE_MAX_SOURCE_LENGTH = 256
SMOKE_MAX_TARGET_LENGTH = 64
STRICT_EXPECTED_ROWS = True
EXPECTED_SPLIT_ROWS = {
    "phase_1": {"train": 10_775, "validation": 1_348, "test": 1_344},
    "phase_2": {"train": 6_909, "validation": 859, "test": 871},
}
# Acceptance gates. 45 ROUGE-L bám theo baseline lịch sử ~48.9 của project;
# đổi các ngưỡng này TRƯỚC khi train nếu sản phẩm có SLA khác.
MIN_PHASE1_ROUGEL = 45.0
MIN_PHASE2_ROUGEL_GAIN = 1.0
MAX_PHASE1_ROUGEL_DROP = 1.0
MAX_VALID_TEST_ROUGEL_GAP = 3.0
RUN_FINAL_TEST = True
KAGGLE_INPUT_SEARCH_DEPTH = 6
SUPPORTED_DATA_SUFFIXES = {".csv", ".parquet", ".pq"}

def _slug_key(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).casefold())

def _split_files(root, split):
    prefix = "valid" if split == "validation" else split
    return sorted(
        path for path in root.glob(f"{prefix}*.*")
        if path.is_file() and path.suffix.casefold() in SUPPORTED_DATA_SUFFIXES
    )

def _is_data_root(root):
    return root.is_dir() and all(_split_files(root, split) for split in ("train", "validation", "test"))

def _discover_data_roots(mount, max_depth=KAGGLE_INPUT_SEARCH_DEPTH):
    discovered = []
    for current_dir, child_dirs, _ in os.walk(mount):
        current = Path(current_dir)
        depth = len(current.relative_to(mount).parts)
        if depth >= max_depth:
            child_dirs[:] = []
        if _is_data_root(current):
            discovered.append(current.resolve())
            # Một data root hợp lệ đã chứa đủ split; không cần dò sâu hơn.
            child_dirs[:] = []
    return list(dict.fromkeys(discovered))

def _matches_phase_name(root, phase):
    root_key = _slug_key(root.as_posix())
    alias_keys = {_slug_key(alias) for alias in KAGGLE_DATASET_ALIASES[phase]}
    return any(alias_key in root_key for alias_key in alias_keys)

def resolve_phase_data_root(phase):
    manual = MANUAL_DATA_ROOTS[phase].strip()
    if manual:
        candidates = [Path(manual).expanduser()]
    elif Path("/kaggle/input").is_dir():
        input_root = Path("/kaggle/input")
        alias_keys = {_slug_key(alias) for alias in KAGGLE_DATASET_ALIASES[phase]}
        named_mounts = [path for path in input_root.iterdir() if path.is_dir() and _slug_key(path.name) in alias_keys]
        candidates = [root for mount in named_mounts for root in _discover_data_roots(mount)]
        if not candidates:
            candidates = [root for mount in input_root.iterdir() if mount.is_dir() for root in _discover_data_roots(mount)]
        named_candidates = [root for root in candidates if _matches_phase_name(root, phase)]
        if named_candidates:
            candidates = named_candidates
    else:
        candidates = [Path("data") / phase]

    candidates = [root for root in candidates if _is_data_root(root)]
    unique = {str(root.resolve()): root.resolve() for root in candidates}
    if len(unique) != 1:
        available = sorted(path.name for path in Path("/kaggle/input").iterdir()) if Path("/kaggle/input").is_dir() else []
        raise RuntimeError(
            f"Không xác định duy nhất data root cho {phase}. discovered={list(unique)}, "
            f"Kaggle inputs={available}. Hãy điền MANUAL_DATA_ROOTS[{phase!r}] bằng thư mục chứa train/validation/test."
        )
    return next(iter(unique.values()))

def build_data_overrides(root):
    def pattern_for(split):
        files = _split_files(root, split)
        if len(files) == 1:
            return str(files[0])
        prefix = "valid" if split == "validation" else split
        suffixes = {path.suffix.casefold() for path in files}
        return str(root / f"{prefix}*{next(iter(suffixes))}") if len(suffixes) == 1 else str(root / f"{prefix}*.*")

    return {
        "data.train_file": pattern_for("train"),
        "data.valid_file": pattern_for("validation"),
        "data.test_file": pattern_for("test"),
    }

PHASE_DATA_ROOTS = {phase: resolve_phase_data_root(phase) for phase in KAGGLE_DATASET_ALIASES}
PHASE_DATA_OVERRIDES = {phase: build_data_overrides(root) for phase, root in PHASE_DATA_ROOTS.items()}

for phase, paths in PHASE_DATA_OVERRIDES.items():
    raw = load_dataset_from_files(paths["data.train_file"], paths["data.valid_file"], paths["data.test_file"])
    counts = {split: len(raw[split]) for split in ("train", "validation", "test")}
    if STRICT_EXPECTED_ROWS and counts != EXPECTED_SPLIT_ROWS[phase]:
        raise RuntimeError(f"Sai số dòng {phase}: expected={EXPECTED_SPLIT_ROWS[phase]}, actual={counts}")
    print(f"✅ {phase}: root={PHASE_DATA_ROOTS[phase]} | rows={counts}")
    del raw
gc.collect()
if SMOKE_TEST:
    print("🧪 RUN_MODE=smoke: chỉ kiểm tra luồng; metric KHÔNG dùng để viết kết luận chất lượng.")
else:
    print("🚀 RUN_MODE=full: chạy train và final quality report đầy đủ.")

## Callback Custom cho Logging

In [ ]:
class KaggleProgressCallback(TrainerCallback):
    def __init__(self, label, heartbeat_seconds=60, log_file=None, raw_dataset=None, tokenizer=None, data_config=None, generation_config=None):
        self.label = label
        self.heartbeat_seconds = max(10, int(heartbeat_seconds))
        self.log_file = Path(log_file) if log_file else None
        self.raw_dataset = raw_dataset
        self.tokenizer = tokenizer
        self.data_config = data_config
        self.generation_config = generation_config
        self.started_at = None
        self.last_completed_step = 0
        self.max_steps = 0
        self._is_main_process = True
        self._stop_event = threading.Event()
        self._write_lock = threading.Lock()
        self._heartbeat_thread = None

    def _device_status(self):
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / (1024 ** 3)
            reserved = torch.cuda.memory_reserved() / (1024 ** 3)
            return f"GPU allocated/reserved={allocated:.2f}/{reserved:.2f} GB"
        return "CPU"

    def _emit(self, message):
        if not self._is_main_process: return
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{timestamp}] [{self.label}] {message}"
        with self._write_lock:
            print(line, flush=True)
            if self.log_file:
                self.log_file.parent.mkdir(parents=True, exist_ok=True)
                with self.log_file.open("a", encoding="utf-8") as handle:
                    handle.write(line + "\n")

    def _heartbeat_loop(self):
        while not self._stop_event.wait(self.heartbeat_seconds):
            self._emit(f"♥ VẪN ĐANG TRAIN | step={self.last_completed_step}/{self.max_steps} | {self._device_status()}")

    def on_train_begin(self, args, state, control, **kwargs):
        self.started_at = time.time()
        self.max_steps = state.max_steps
        self._is_main_process = state.is_world_process_zero
        self._stop_event.clear()
        if self._is_main_process:
            self._heartbeat_thread = threading.Thread(target=self._heartbeat_loop, daemon=True)
            self._heartbeat_thread.start()

    def on_step_end(self, args, state, control, **kwargs):
        self.last_completed_step = state.global_step

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        fields = [f"{k}={v:.6g}" if isinstance(v, float) else f"{k}={v}" for k, v in logs.items() if k in ("loss", "learning_rate", "epoch")]
        if fields: self._emit("METRICS | " + " | ".join(fields))

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        metrics = metrics or {}
        if self.log_file:
            csv_path = self.log_file.parent / "eval_history.csv"
            row = { "step": state.global_step, "epoch": round(state.epoch, 2) if state.epoch else 0, **{k:v for k,v in metrics.items() if k.startswith("eval_")} }
            write_header = not csv_path.exists()
            with open(csv_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=row.keys())
                if write_header: writer.writeheader()
                writer.writerow(row)

        summary = ", ".join(f"{k}={v:.4f}" for k, v in metrics.items() if k in ("eval_loss", "eval_rougeL", "eval_gen_len"))
        self._emit(f"ĐÁNH GIÁ XONG | {summary}")

        # In mẫu sinh text
        if self.raw_dataset is not None and self.tokenizer and kwargs.get('model'):
            self._print_samples(kwargs['model'])

    def _print_samples(self, model, n=2):
        was_training = model.training
        model.eval()
        device = next(model.parameters()).device
        for i in range(min(n, len(self.raw_dataset))):
            article = self.raw_dataset[i]["article"]
            ref = self.raw_dataset[i]["summary"]
            source = (self.data_config.source_prefix or "") + article
            inputs = self.tokenizer(source, max_length=self.data_config.max_source_length, truncation=True, return_tensors="pt").to(device)
            with torch.no_grad():
                out = model.generate(**inputs, max_length=self.generation_config.max_length, num_beams=self.generation_config.num_beams)
            pred = self.tokenizer.decode(out[0], skip_special_tokens=True)
            self._emit(f"\n--- Mẫu {i+1} ---\n📄 Gốc: {article[:150]}...\n✅ Ref: {ref}\n🤖 Gen: {pred}")
        if was_training:
            model.train()

    def stop(self):
        self._stop_event.set()
        if self._heartbeat_thread and self._heartbeat_thread.is_alive():
            self._heartbeat_thread.join(timeout=2)
            
    def on_train_end(self, args, state, control, **kwargs):
        self._emit("KẾT THÚC TRAIN")
        self.stop()


## Hàm Helper Train Phase

In [ ]:
def run_phase(config_file, data_phase, model_override=None, output_override=None, resume_from_checkpoint=None):
    print(f"\n{'='*80}\n🚀 BẮT ĐẦU: {config_file}\n{'='*80}")
    config = load_config(config_file)
    
    if data_phase not in PHASE_DATA_OVERRIDES:
        raise ValueError(f"data_phase không hợp lệ: {data_phase}")
    overrides = dict(PHASE_DATA_OVERRIDES[data_phase])
    if model_override: overrides["model.name_or_path"] = model_override
    requested_output = Path(output_override or config.training.output_dir)
    if SMOKE_TEST and output_override is None:
        requested_output = Path("smoke_outputs") / SMOKE_RUN_TAG / config.phase.name
    if Path("/kaggle").exists() and not requested_output.is_absolute():
        requested_output = Path("/kaggle/working") / requested_output
    overrides["training.output_dir"] = str(requested_output)
    if SMOKE_TEST:
        overrides.update({
            "data.max_train_samples": SMOKE_TRAIN_SAMPLES,
            "data.max_eval_samples": SMOKE_EVAL_SAMPLES,
            "data.max_source_length": SMOKE_MAX_SOURCE_LENGTH,
            "data.max_target_length": SMOKE_MAX_TARGET_LENGTH,
            "training.max_steps": SMOKE_MAX_STEPS,
            "training.per_device_train_batch_size": 1,
            "training.per_device_eval_batch_size": 1,
            "training.gradient_accumulation_steps": 1,
            "training.eval_strategy": "steps",
            "training.eval_steps": 1,
            "training.save_strategy": "steps",
            "training.save_steps": 1,
            "training.save_total_limit": 1,
            "training.logging_steps": 1,
            "training.early_stopping_patience": 0,
            "generation.max_length": SMOKE_MAX_TARGET_LENGTH,
            "generation.num_beams": 1,
            "generation.early_stopping": False,
        })
    if resume_from_checkpoint:
        overrides["training.resume_from_checkpoint"] = resume_from_checkpoint
    
    config = apply_overrides(config, overrides)
    output_dir = Path(config.training.output_dir)
    best_dir = output_dir / "best"
    output_dir.mkdir(parents=True, exist_ok=True)
    save_json(config_to_dict(config), output_dir / "resolved_config.json")
    
    set_seed(config.training.seed)
    tokenizer = load_tokenizer(config.model)
    model = load_model(config.model, tokenizer, config.generation)
    
    if config.training.gradient_checkpointing:
        enable_gradient_checkpointing(model)
    if config.training.freeze_encoder:
        freeze_encoder(model)
    
    # Test là holdout: không tải/tokenize trong quá trình chọn checkpoint.
    training_data_config = replace(config.data, test_file="")
    datasets = load_and_preprocess(tokenizer, training_data_config)
    raw_valid_dataset = load_dataset_from_files(valid_file=config.data.valid_file)["validation"]
    
    tc = config.training
    training_args = build_training_args(config)
    
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100)
    
    callbacks = [KaggleProgressCallback(
        label=config.phase.name,
        log_file=output_dir / "training.log",
        raw_dataset=raw_valid_dataset,
        tokenizer=tokenizer,
        data_config=config.data,
        generation_config=config.generation,
    )]
    if tc.early_stopping_patience > 0:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=tc.early_stopping_patience))
        
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=datasets["train"],
        eval_dataset=datasets["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=build_compute_metrics(tokenizer),
        callbacks=callbacks,
    )
    
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    
    try:
        train_result = trainer.train(resume_from_checkpoint=tc.resume_from_checkpoint)
    finally:
        callbacks[0].stop()
        
    trainer.save_model(str(best_dir))
    tokenizer.save_pretrained(str(best_dir))
    
    eval_res = trainer.evaluate(metric_key_prefix="eval")
    save_json(train_result.metrics, output_dir / "train_results.json")
    save_json(eval_res, output_dir / "eval_results.json")
    
    del trainer, model, datasets, raw_valid_dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return str(best_dir.resolve()), eval_res


## 🏃‍♂️ Chạy Phase 1: Full Fine-tune (General Data)

In [ ]:
best_phase1, eval_p1 = run_phase("configs/vit5_base_phase_1.yaml", data_phase="phase_1")
print(f"Phase 1 hoàn thành. Checkpoint: {best_phase1}")
print(f"ROUGE-L Phase 1: {eval_p1['eval_rougeL']}")

## 🏃‍♂️ Chạy Phase 2a: Freeze Encoder (Medical Data)

In [ ]:
best_phase2a, eval_p2a = run_phase(
    "configs/vit5_base_phase_2a.yaml",
    data_phase="phase_2",
    model_override=best_phase1
)
print(f"Phase 2a hoàn thành. Checkpoint: {best_phase2a}")
print(f"ROUGE-L Phase 2a: {eval_p2a['eval_rougeL']}")

## 🏃‍♂️ Chạy Phase 2b: Unfreeze All, Low LR (Medical Data)

In [ ]:
best_phase2b, eval_p2b = run_phase(
    "configs/vit5_base_phase_2b.yaml",
    data_phase="phase_2",
    model_override=best_phase2a
)
print(f"Phase 2b hoàn thành. Checkpoint: {best_phase2b}")
print(f"ROUGE-L Phase 2b: {eval_p2b['eval_rougeL']}")

## 🎯 Validation selection → khóa checkpoint → final test

Cell này tạo bảng so sánh Phase 1/2a/2b trên cả hai validation split, chỉ chọn model đạt adaptation + retention gate, rồi mới mở test split. Kết luận là **gate kỹ thuật**, không thay thế kiểm tra factuality thủ công từ các file prediction.

In [ ]:
from src.evaluator import evaluate_checkpoint
import pandas as pd

working_root = Path("/kaggle/working") if Path("/kaggle").exists() else Path(".")
evaluation_root = working_root / "evaluations_progressive"
if SMOKE_TEST:
    evaluation_root = working_root / "smoke_evaluations_progressive" / SMOKE_RUN_TAG
evaluation_root.mkdir(parents=True, exist_ok=True)

phase1_eval_overrides = dict(PHASE_DATA_OVERRIDES["phase_1"])
phase2_eval_overrides = dict(PHASE_DATA_OVERRIDES["phase_2"])
if SMOKE_TEST:
    smoke_eval_overrides = {
        "data.max_eval_samples": SMOKE_EVAL_SAMPLES,
        "data.max_source_length": SMOKE_MAX_SOURCE_LENGTH,
        "data.max_target_length": SMOKE_MAX_TARGET_LENGTH,
        "training.per_device_eval_batch_size": 1,
        "generation.max_length": SMOKE_MAX_TARGET_LENGTH,
        "generation.num_beams": 1,
        "generation.early_stopping": False,
    }
    phase1_eval_overrides.update(smoke_eval_overrides)
    phase2_eval_overrides.update(smoke_eval_overrides)

config_phase1 = apply_overrides(
    load_config("configs/vit5_base_phase_1.yaml"),
    phase1_eval_overrides,
)
config_phase2 = apply_overrides(
    load_config("configs/vit5_base_phase_2b.yaml"),
    phase2_eval_overrides,
)

def run_evaluation(label, model_path, config, data_phase, split, export_predictions=False):
    print(f"\n🔎 {label}: {model_path} trên {data_phase}/{split}")
    return evaluate_checkpoint(
        model_path=model_path,
        config=config,
        output_dir=evaluation_root / f"{label}_on_{data_phase}_{split}",
        export_predictions=export_predictions,
        split=split,
    )

def rouge_l(metrics, prefix):
    return float(metrics[f"{prefix}_rougeL"])

def detailed_row(model_name, dataset_name, metrics, prefix):
    return {
        "Model": model_name,
        "Dataset": dataset_name,
        "Loss": round(float(metrics[f"{prefix}_loss"]), 4),
        "ROUGE-1": round(float(metrics[f"{prefix}_rouge1"]), 2),
        "ROUGE-2": round(float(metrics[f"{prefix}_rouge2"]), 2),
        "ROUGE-L": round(float(metrics[f"{prefix}_rougeL"]), 2),
        "Gen length": round(float(metrics[f"{prefix}_gen_len"]), 1),
    }

print("\n=== 1. VALIDATION MATRIX (dùng để chọn model) ===")
phase1_on_phase2_val_metrics = run_evaluation(
    "phase1", best_phase1, config_phase2, "phase2", "validation"
)
phase2a_on_phase1_val_metrics = run_evaluation(
    "phase2a", best_phase2a, config_phase1, "phase1", "validation"
)
phase2b_on_phase1_val_metrics = run_evaluation(
    "phase2b", best_phase2b, config_phase1, "phase1", "validation"
)

phase1_on_phase1_val = float(eval_p1["eval_rougeL"])
phase1_on_phase2_val = rouge_l(phase1_on_phase2_val_metrics, "validation")
phase2a_on_phase1_val = rouge_l(phase2a_on_phase1_val_metrics, "validation")
phase2a_on_phase2_val = float(eval_p2a["eval_rougeL"])
phase2b_on_phase1_val = rouge_l(phase2b_on_phase1_val_metrics, "validation")
phase2b_on_phase2_val = float(eval_p2b["eval_rougeL"])

phase1_quality_pass = phase1_on_phase1_val >= MIN_PHASE1_ROUGEL
candidates = [
    {
        "name": "phase2a",
        "checkpoint": best_phase2a,
        "phase1_validation_rougeL": phase2a_on_phase1_val,
        "phase2_validation_rougeL": phase2a_on_phase2_val,
    },
    {
        "name": "phase2b",
        "checkpoint": best_phase2b,
        "phase1_validation_rougeL": phase2b_on_phase1_val,
        "phase2_validation_rougeL": phase2b_on_phase2_val,
    },
]

for candidate in candidates:
    candidate["retention_delta"] = candidate["phase1_validation_rougeL"] - phase1_on_phase1_val
    candidate["phase2_gain"] = candidate["phase2_validation_rougeL"] - phase1_on_phase2_val
    candidate["retention_pass"] = candidate["retention_delta"] >= -MAX_PHASE1_ROUGEL_DROP
    candidate["adaptation_pass"] = candidate["phase2_gain"] >= MIN_PHASE2_ROUGEL_GAIN
    candidate["validation_pass"] = bool(
        phase1_quality_pass and candidate["retention_pass"] and candidate["adaptation_pass"]
    )

validation_rows = [
    {
        "Model": "phase1",
        "P1 ROUGE-L": phase1_on_phase1_val,
        "P2 ROUGE-L": phase1_on_phase2_val,
        "Retention Δ": 0.0,
        "P2 gain": 0.0,
        "Pass": phase1_quality_pass,
    }
]
validation_rows.extend(
    {
        "Model": item["name"],
        "P1 ROUGE-L": item["phase1_validation_rougeL"],
        "P2 ROUGE-L": item["phase2_validation_rougeL"],
        "Retention Δ": item["retention_delta"],
        "P2 gain": item["phase2_gain"],
        "Pass": item["validation_pass"],
    }
    for item in candidates
)
validation_table = pd.DataFrame(validation_rows)
for column in ("P1 ROUGE-L", "P2 ROUGE-L", "Retention Δ", "P2 gain"):
    validation_table[column] = validation_table[column].round(2)
print(validation_table.to_string(index=False))
print(
    f"\nGate: P1 ROUGE-L >= {MIN_PHASE1_ROUGEL:.2f}; "
    f"P2 gain >= {MIN_PHASE2_ROUGEL_GAIN:.2f}; "
    f"P1 drop <= {MAX_PHASE1_ROUGEL_DROP:.2f}."
)

eligible = [item for item in candidates if item["validation_pass"]]
if SMOKE_TEST:
    selected = max(candidates, key=lambda item: item["phase2_validation_rougeL"])
    print(f"🧪 Smoke mode chọn tạm {selected['name']} chỉ để test tiếp luồng test/report.")
else:
    selected = max(eligible, key=lambda item: item["phase2_validation_rougeL"]) if eligible else None
selection_report = {
    "run_mode": RUN_MODE,
    "quality_metrics_valid": not SMOKE_TEST,
    "thresholds": {
        "min_phase1_rougeL": MIN_PHASE1_ROUGEL,
        "min_phase2_rougeL_gain": MIN_PHASE2_ROUGEL_GAIN,
        "max_phase1_rougeL_drop": MAX_PHASE1_ROUGEL_DROP,
        "max_validation_test_rougeL_gap": MAX_VALID_TEST_ROUGEL_GAP,
    },
    "phase1_validation_pass": bool(phase1_quality_pass),
    "phase1_on_phase1_validation_rougeL": phase1_on_phase1_val,
    "phase1_on_phase2_validation_rougeL": phase1_on_phase2_val,
    "candidates": candidates,
    "selected": selected,
}
save_json(selection_report, evaluation_root / "validation_selection.json")

if selected is None:
    print("\n❌ CHƯA ĐẠT VALIDATION GATE: không mở test và không chọn Phase 2 checkpoint.")
    print("Hãy giữ Phase 1 cho general; thử LoRA/replay hoặc tune lại bằng validation, không tune trên test.")
elif not RUN_FINAL_TEST:
    if SMOKE_TEST:
        print("\n🧪 Smoke train/validation đã chạy xong; RUN_FINAL_TEST=False nên chưa test nhánh final report.")
    else:
        print(f"\n✅ ĐẠT VALIDATION GATE. Đã khóa checkpoint: {selected['name']} -> {selected['checkpoint']}")
        print("RUN_FINAL_TEST=False nên chưa mở test split.")
else:
    print(f"\n🔒 Đã khóa checkpoint trước test: {selected['name']} -> {selected['checkpoint']}")
    print("\n=== 2. FINAL TEST MATRIX (không dùng để chọn/tune checkpoint) ===")
    phase1_on_phase1_test_metrics = run_evaluation(
        "phase1", best_phase1, config_phase1, "phase1", "test", export_predictions=True
    )
    phase1_on_phase2_test_metrics = run_evaluation(
        "phase1", best_phase1, config_phase2, "phase2", "test", export_predictions=True
    )
    selected_on_phase1_test_metrics = run_evaluation(
        selected["name"], selected["checkpoint"], config_phase1, "phase1", "test", export_predictions=True
    )
    selected_on_phase2_test_metrics = run_evaluation(
        selected["name"], selected["checkpoint"], config_phase2, "phase2", "test", export_predictions=True
    )

    test_table = pd.DataFrame([
        detailed_row("phase1", "phase1", phase1_on_phase1_test_metrics, "test"),
        detailed_row("phase1", "phase2", phase1_on_phase2_test_metrics, "test"),
        detailed_row(selected["name"], "phase1", selected_on_phase1_test_metrics, "test"),
        detailed_row(selected["name"], "phase2", selected_on_phase2_test_metrics, "test"),
    ])
    print(test_table.to_string(index=False))

    phase1_on_phase1_test = rouge_l(phase1_on_phase1_test_metrics, "test")
    phase1_on_phase2_test = rouge_l(phase1_on_phase2_test_metrics, "test")
    selected_on_phase1_test = rouge_l(selected_on_phase1_test_metrics, "test")
    selected_on_phase2_test = rouge_l(selected_on_phase2_test_metrics, "test")
    test_retention_delta = selected_on_phase1_test - phase1_on_phase1_test
    test_phase2_gain = selected_on_phase2_test - phase1_on_phase2_test
    test_checks = {
        "phase1_quality": phase1_on_phase1_test >= MIN_PHASE1_ROUGEL,
        "phase1_generalization": phase1_on_phase1_test >= phase1_on_phase1_val - MAX_VALID_TEST_ROUGEL_GAP,
        "selected_phase1_generalization": selected_on_phase1_test >= selected["phase1_validation_rougeL"] - MAX_VALID_TEST_ROUGEL_GAP,
        "selected_phase2_generalization": selected_on_phase2_test >= selected["phase2_validation_rougeL"] - MAX_VALID_TEST_ROUGEL_GAP,
        "retention": test_retention_delta >= -MAX_PHASE1_ROUGEL_DROP,
        "phase2_adaptation": test_phase2_gain >= MIN_PHASE2_ROUGEL_GAIN,
    }
    measured_quality_pass = bool(selected["validation_pass"] and all(test_checks.values()))
    final_pass = bool(not SMOKE_TEST and measured_quality_pass)
    final_report = {
        **selection_report,
        "test": {
            "phase1_on_phase1": phase1_on_phase1_test_metrics,
            "phase1_on_phase2": phase1_on_phase2_test_metrics,
            "selected_on_phase1": selected_on_phase1_test_metrics,
            "selected_on_phase2": selected_on_phase2_test_metrics,
            "retention_delta": test_retention_delta,
            "phase2_gain": test_phase2_gain,
            "checks": test_checks,
        },
        "smoke_flow_completed": bool(SMOKE_TEST),
        "measured_quality_pass": measured_quality_pass,
        "final_pass": final_pass,
    }
    save_json(final_report, evaluation_root / "final_quality_report.json")

    print("\n=== 3. KẾT LUẬN ===")
    print(f"Test retention Δ: {test_retention_delta:+.2f} ROUGE-L")
    print(f"Test Phase 2 gain: {test_phase2_gain:+.2f} ROUGE-L")
    for check, passed in test_checks.items():
        print(f"{'✅' if passed else '❌'} {check}")
    if SMOKE_TEST:
        print("\n✅ SMOKE TEST HOÀN TẤT TOÀN BỘ LUỒNG.")
        print("Các metric trên 8 mẫu/2 steps KHÔNG phải kết luận chất lượng.")
        print("Đổi RUN_MODE='full', Restart Session rồi Run All để train thật.")
    elif final_pass:
        print(f"\n✅ ĐẠT GATE KỸ THUẬT — dùng checkpoint {selected['checkpoint']}")
    else:
        print("\n❌ CHƯA ĐẠT GATE KỸ THUẬT — không deploy checkpoint progressive này.")
    print("ROUGE không đo factuality; hãy đọc predictions_test.jsonl trước khi kết luận production-ready.")
